# Inspect a graph Scenario

Load a graph-mode scenario script and walk every inspection surface before
running the simulation.

**Graph-mode scenarios** carry `nodes: list[NodeInstance]` and
`edges: list[EdgeSpec]` instead of the legacy `stores` list. The `is_graph`
property is `True` when `nodes` is non-empty. Two new inspection views are
available: `Scenario.nodes_df()` and `Scenario.edges_df()`.

This notebook uses `scenarios/example_homogeneous.py` (no OpenAI key required).

See `notebooks/01-openai_world_builder.ipynb` for the quickstart 3-node chain,
and `notebooks/02-inspect_world.ipynb` for World artifact inspection.

In [ ]:
%load_ext autoreload
%autoreload 2

import os
from pathlib import Path

while not (Path.cwd() / 'pyproject.toml').exists():
    os.chdir('..')

import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 160)

from src.sim.scenario import load_scenario_from_path

## Load a scenario

Point `SCENARIO_PATH` at any scenario script that exposes a top-level
`scenario` attribute. The homogeneous example uses a hand-authored catalog
and requires no LLM or API key.

In [ ]:
SCENARIO_PATH = 'scenarios/example_homogeneous.py'

scenario = load_scenario_from_path(SCENARIO_PATH)
print(f'Loaded scenario: {len(scenario.catalog)} catalog items, '
      f'{len(scenario.nodes)} node instances, '
      f'{len(scenario.edges)} edges, '
      f'{scenario.n_steps} steps')
print(f'is_graph: {scenario.is_graph}')

### Summary — top-level simulation parameters

One-row overview: step count, start date, world seed, and the size of the
catalog and node roster. Use this as a quick sanity check.

In [ ]:
scenario.summary_df()

### Nodes — the graph actors

`scenario.nodes_df()` returns one row per `NodeInstance`. Columns include:
`node_id`, `node_type` (`FactoryNode` / `IntermediateNode` / `DemandSinkNode`),
`region`, `init_seed`, `level` (echelon level, 0 = factory), and type-specific
fields (`product_id` for sinks, `carried_products` for intermediates, etc.).

`policy_class` is populated from the live policy attached by the scenario
script — `None` when the scenario is loaded from a serialised JSON artifact
(policies are not serialised by design).

In [ ]:
nodes_df = scenario.nodes_df()
nodes_df

In [ ]:
# Count by node type
nodes_df['node_type'].value_counts()

In [ ]:
# Policy class per node type — confirm all shops share one policy class
nodes_df.groupby('node_type')['policy_class'].value_counts()

### Edges — supply links between nodes

`scenario.edges_df()` returns one row per `EdgeSpec`. Columns: `supplier_id`,
`buyer_id`, `default_lead_time`, `per_product_lead_time` (dict of per-product
overrides, or `None` when the default applies to all products).

In [ ]:
edges_df = scenario.edges_df()
edges_df

In [ ]:
# Lead time distribution
edges_df['default_lead_time'].value_counts().rename('edge_count').to_frame()

### Catalog — items for sale

One row per SKU. `freshness_alpha` and `freshness_decay` govern the hype-curve
multiplier inside `DemandSinkNode.demand_target`.

In [ ]:
scenario.catalog_df()

### Market — demand and seasonality parameters

In [ ]:
scenario.market_df().T

### Disruption — supply-chain shock parameters

Governs random disruption events: `event_prob` is the per-step probability,
`types` lists the event categories, `severity` and `duration` are
`Distribution` objects sampled at event time.

In [ ]:
scenario.disruption_df()

### Lifecycle — product stage defaults

In [ ]:
scenario.lifecycle_df()

## Visualise the graph topology

Build a simple adjacency list from `edges_df` and print the factory→shop→sink chains.

In [ ]:
from collections import defaultdict

children: dict[str, list[str]] = defaultdict(list)
for _, row in edges_df.iterrows():
    children[row['supplier_id']].append(row['buyer_id'])

# Find roots (nodes that never appear as a buyer)
all_buyers = set(edges_df['buyer_id'])
roots = [ni.node.id for ni in scenario.nodes if ni.node.id not in all_buyers]

def print_tree(node_id: str, prefix: str = "", last: bool = True):
    connector = '└── ' if last else '├── '
    print(prefix + connector + node_id)
    child_ids = children.get(node_id, [])
    for i, child in enumerate(child_ids):
        new_prefix = prefix + ('    ' if last else '│   ')
        print_tree(child, new_prefix, i == len(child_ids) - 1)

for root in sorted(roots):
    print_tree(root)

## Run the scenario

In [ ]:
from src.sim.runner import Runner

run_log = Runner(scenario).run()
print(f'Run log: {run_log["n_steps"]} steps, {len(run_log["ticks"])} tick records')
print(f'Global keys: {list(run_log["global"].keys())}')

In [ ]:
import matplotlib.pyplot as plt
from src.sim.node import IntermediateNode

shop_ids = sorted(ni.node.id for ni in scenario.nodes if isinstance(ni.node, IntermediateNode))

cash_by_shop = {sid: [t['node_cash'].get(sid, 0.0) for t in run_log['ticks']] for sid in shop_ids}

fig, ax = plt.subplots(figsize=(12, 4))
for sid, vals in cash_by_shop.items():
    ax.plot(vals, label=sid)
ax.set_title('Shop cash over time (example_homogeneous)')
ax.set_xlabel('tick')
ax.set_ylabel('cash (£)')
ax.legend(bbox_to_anchor=(1.01, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()

## (Optional) Historical-run audit via `Scenario.from_json`

After running a simulation, the exporter serialises the scenario to
`data/<run>/config/scenario.json`. Load it back with `Scenario.from_json` to
inspect the exact graph topology used in a past run.

**Note:** `policy_class` is `None` in `nodes_df()` of a deserialised scenario
— this is by design; policies are not serialised.

In [ ]:
from src.sim.scenario import Scenario

SCENARIO_JSON_PATH = Path('data/example_homogeneous/config/scenario.json')

if SCENARIO_JSON_PATH.exists():
    scenario_from_run = Scenario.from_json(SCENARIO_JSON_PATH.read_text())
    print('Loaded from historical run artifact')
    print(f'is_graph: {scenario_from_run.is_graph}')
    print(f'nodes: {len(scenario_from_run.nodes)}, edges: {len(scenario_from_run.edges)}')
    display(scenario_from_run.nodes_df())
    print('\nNote: policy_class is None by design — policies are not serialised.')
else:
    print(f'Artifact not found: {SCENARIO_JSON_PATH}')
    print('Run `uv run python main.py scenarios/example_homogeneous.py` to generate it.')